In [ ]:
# imports

# Auto-reload modules before executing code
%load_ext autoreload
%autoreload 2

import echopype as ep
from pathlib import Path
import numpy as np

from aa_si_calibration import calibration
from aa_si_calibration import comparison
from aa_si_calibration import standardized_file_lib
from aa_si_calibration import manufacturer_file_parsers

from aa_si_ml import ml
from aa_si_utils import utils
from aa_si_visualization import assorted

import pyvista as pv

import xarray as xr

import yaml
import jsonschema

In [ ]:
# User defined variables
# NOTE: must include a .bot file with the same name in the same directory as the raw file!
raw_folder_string = './NEFSC_use_case_1_raw_files/' # NOTE: this is now a folder and not a full path to a single raw file
raw_file_names = [] # Only use if populated with a list strings of raw file names
netcdf_output_folder_string = './NetCDF-files'
sv_output_folder_string = './Sv-files'
cal_folder_string = '../calibration_files/HB201607_cal'
output_logs_folder_string = './Output-Logs'
clear_previous_json_logs = True
standardized_calibration_from_report = "HB1603_L1-D20160703_cal_report.yml"
standardized_calibration_from_nc = "HB1603_L1-D20160703_cal_nc_params.yml"
standardized_cal_files_location = "./standardized_cal_files"

standardized_cal_folder = Path(standardized_cal_files_location)
raw_folder = Path(raw_folder_string)
netcdf_output_folder = Path(netcdf_output_folder_string)
netcdf_output_path = netcdf_output_folder / (raw_folder.stem + '.nc') # NOTE: need to calculate this for every raw file if multiple files
sv_output_folder = Path(sv_output_folder_string)
cal_folder = Path(cal_folder_string)
output_logs_folder = Path(output_logs_folder_string)

sv_flag_thresholds = {
                "critical_median": 2.0,
                "large_median": 1.0,
                "moderate_median": 0.5,
                "critical_max": 4.0,
                "large_max": 2.0,
                "moderate_max": 1.0
            }

# Get list of raw files to process
if raw_file_names is not None and len(raw_file_names) > 0:
    # Use specified file names
    raw_file_paths = [raw_folder / filename for filename in raw_file_names]
else:
    # Get all .raw files in folder
    raw_file_paths = sorted(raw_folder.glob("*.raw"))

if not raw_file_paths:
    raise Exception("No raw files found to process")

print(f"Found {len(raw_file_paths)} raw file(s) to process:")
for path in raw_file_paths:
    print(f"  - {path.name}")

if not cal_folder.exists():
    raise Exception("Calibration folder not found")

if not raw_folder.exists():
    raise Exception("Raw folder not found")

# Ensure output folders exist
output_folders = [
    netcdf_output_folder,
    sv_output_folder,
    output_logs_folder,
    standardized_cal_folder,
]
for folder in output_folders:
    if not folder.exists():
        folder.mkdir(parents=True, exist_ok=True)
        print(f"Created missing output folder: {folder}")

if clear_previous_json_logs:
    flags_file = Path(output_logs_folder) / "calibration_flags_NEFSC_use_case_1.json"
    if flags_file.exists():
        flags_file.unlink()

In [ ]:
# Open the raw files and convert to netCDF
# Processing Level 1

echodata = utils.open_and_combine_raw_files(raw_file_paths, netcdf_output_folder, sonar_model="EK60", include_bot=True)

In [ ]:
# extract calibration parameters from original netcdf file
params = calibration.extract_netcdf_calibration_parameters(echodata, output_logs_folder)

original_cal_params = params["cal_params"]
original_env_params = params["env_params"]
original_other_params = params["other_params"]

# the calibration parameters come from the first file in the sequential raw files that were combined by echopype
nc_cal_files = [str(raw_file_paths[0].name)]

original_other_params["source_filenames_across_channels"] = nc_cal_files
original_other_params["source_file_type"] = ".raw"

In [ ]:
# print parameters from netcdf file
calibration.print_calibration_values(echodata, original_env_params, original_cal_params, original_other_params, "Calibration Values From .nc File")

In [ ]:
# extract calibration parameters from EK60 cal file and convert to pipeline format

report_params = manufacturer_file_parsers.extract_calibration_params_from_EK60_report(
    cal_folder,
    original_other_params["frequency_nominal"],
    output_logs_folder
    )

report_cal_params, report_env_params, report_other_params = manufacturer_file_parsers.convert_ek60_params_to_pipeline_format(report_params)

In [ ]:
# print file calibration parameters
calibration.print_calibration_values(echodata, report_env_params, report_cal_params, report_other_params, "Calibration Values From .cal File Report in .nc format")

In [ ]:
global_params_report = {
    "cruise_id" : "HB1603"
}

file_path_report = standardized_cal_folder / standardized_calibration_from_report

standardized_file_lib.save_cal_params_to_standardized_file(report_cal_params, report_env_params, report_other_params, global_params_report, file_path_report)

In [ ]:
global_params_nc = {
    "cruise_id" : "HB1603"
}

file_path_nc = standardized_cal_folder / standardized_calibration_from_nc

standardized_file_lib.save_cal_params_to_standardized_file(original_cal_params, original_env_params, original_other_params, global_params_nc, standardized_cal_file_path=file_path_nc)

In [ ]:
# Run full calibration comparison pipeline
# Compares parameters, computes baseline/calibrated Sv, analyzes individual and combined effects,
# generates diagnostic plots, and verifies additive effects.

comparison_results = comparison.run_full_calibration_comparison(
    echodata,
    report_cal_params,
    report_env_params,
    report_other_params,
    original_cal_params,
    original_env_params,
    original_other_params,
    output_logs_folder,
    sv_output_folder,
    sv_flag_thresholds=sv_flag_thresholds,
)

ds_Sv_baseline = comparison_results["ds_Sv_baseline"]
ds_Sv_combined_test = comparison_results["ds_Sv_combined_test"]
mask = comparison_results["mask"]
verification_results = comparison_results["verification_results"]

In [ ]:
print(ds_Sv_baseline)

In [ ]:
ping = 679

sv_at_ping = ds_Sv_baseline['Sv'].isel(channel=0, ping_time=ping)

# print(sv_at_ping)

valid_indicies = ~np.isnan(sv_at_ping)

# print(valid_indicies)

valid_depths = ds_Sv_baseline["range_sample"].values[valid_indicies]

# print(valid_depths)

last_depth = valid_depths[-1]

print(ds_Sv_baseline["ping_time"][ping].values)
print(last_depth)

print(ds_Sv_baseline['ping_time'])

In [ ]:
ds_Sv_clean, ds_MVBS = ml.data_preprocessing_pipeline(
    ds_Sv_baseline,
    echodata,
    noise_range_sample_num=10,
    noise_ping_num=5,
    mvbs_range_bin="2m",
    mvbs_ping_time_bin="10s",
    mvbs_nan_threshold=.6,
    plot_window=[0, 1200, 0, 679],
    )

In [ ]:
print(ds_MVBS)

In [ ]:
cluster_colors = [
                "#FF8800", "#35E200", "#FF0000", "#2F00FF", "#FF2DF4",
                "#FFFB1C", "#000F92", "#970021", "#8E00E0", "#017685FF", "#4100B9FF"
            ]

In [ ]:
dataset_name = "ml_dataset_2"
custom_normalization_name = "normalized_data_flatten_mean_centered"
ml_result_name = "dbscan_background_results_2"

ds_MVBS, gridded_background_results_dbscan, dbscan_results = ml.full_dbscan_iteration(
    ds_MVBS,
    dataset_name,
    ds_Sv_clean,
    feature_strategy="mean_centered",
    data_var="Sv",
    custom_normalization_name=custom_normalization_name,
    ml_result_name=ml_result_name,
    normalization_strategy="flatten",
    plot_window=[0, 1200, 0, 600],
    min_samples=2,
    sample_size=1000000,
    min_cluster_size=800,
    cluster_selection_method="eom", # also leaf
    use_hdbscan=True,
    soft_membership_threshold=.6
    )
